# AE3 — Transformación y enriquecimiento de datos

**Proyecto Integrado III — Analítica de Datos** · PREICA2602B020038
**Estudiante:** Jesús González · **Docente:** Andrés Felipe Callejas

Este cuaderno continúa el trabajo de la entrega anterior sobre el mismo dataset
(`scikit-learn/iris`, descargado de Hugging Face). Aquí el dataset se prepara
mediante transformaciones validadas, se enriquece con variables calculadas y se
usa para responder las preguntas de negocio del proyecto. Los resultados quedan
en tablas `.csv` y el resumen descriptivo del DataFrame final se exporta a un
archivo `.txt` generado por código.

Todas las rutas del cuaderno son relativas a la carpeta `AE3/`, de modo que
cualquier persona puede reproducirlo con *Run All* después de instalar las
dependencias de `requirements.txt`.

## 1. Necesidad, preguntas de negocio y variables

La entrega anterior se concentró en la carga del dataset y en las clases de
apoyo, y no dejó formalizadas las preguntas de negocio del proyecto. Para no
avanzar con un análisis sin rumbo, esta etapa las define y las deja registradas
como parte del alcance: es el ajuste que se justifica frente al plan original,
pues la necesidad existía desde el inicio pero todavía no estaba escrita.

**Necesidad del proyecto.** El proyecto necesita una caracterización descriptiva
de las tres especies de *Iris* a partir de sus medidas morfológicas, que sirva
para apoyar la identificación de ejemplares a partir de mediciones simples de
sépalo y pétalo, y que deje las tablas listas para la presentación gráfica de la
siguiente etapa. El dataset es público (las medidas provienen del trabajo clásico
de Fisher sobre flores de iris recolectadas por Edgar Anderson), así que no hay
restricción para publicarlo completo en el repositorio.

Las cuatro preguntas que orientan el análisis son las siguientes.

| # | Pregunta de negocio | Variables necesarias | Resultado esperado |
| --- | --- | --- | --- |
| P1 | ¿Cómo se caracteriza morfológicamente cada especie? | `especie` y las cuatro medidas (largo y ancho de sépalo y pétalo) | Tabla con media, mediana y desviación de cada medida por especie |
| P2 | ¿Las especies se diferencian más por el sépalo o por el pétalo? | Las cuatro medidas, agrupadas por especie | Comparación de la variación dentro de cada especie y de la distancia entre especies |
| P3 | ¿Qué relación existe entre las medidas del sépalo y las del pétalo? | Las cuatro medidas, con correlaciones globales y por especie | Tabla de correlaciones de Pearson que indique si una medida predice a la otra |
| P4 | ¿Cómo se distribuyen los ejemplares según el tamaño de la flor? | `especie`, área del pétalo (calculada) y categoría de tamaño (calculada) | Tablas de conteo y porcentaje por categoría y cruce especie–tamaño |

Si alguna pregunta no pudiera responderse por falta de datos, se declarará la
limitación en su sección en lugar de forzar una respuesta.

## 2. Carga del dataset y versión sin modificar

Se usa la clase `HFDatasetLoader` construida en la entrega anterior. Antes de
cualquier operación se conserva una copia intacta del DataFrame, que servirá al
final para comparar qué cambió y qué no.

In [1]:
import os
import warnings

os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")
# La barra de progreso de tqdm no aporta nada en un cuaderno ejecutado de una vez;
# se silencia su aviso sobre ipywidgets para mantener limpia la salida.
warnings.filterwarnings("ignore", message=".*IProgress not found.*")

from pathlib import Path
import sys

import pandas as pd

RUTA_BASE = Path.cwd()          # carpeta AE3/
sys.path.append(str(RUTA_BASE / "src"))

from src.config import DATOS_DIR, RESULTADOS_DIR, TARGET
from src.dataset_summarizer import DatasetSummarizer
from src.hf_loader import HFDatasetLoader

cargador = HFDatasetLoader(TARGET.dataset_id, split=TARGET.split)
df_original = cargador.load()
print(cargador.info)
print()
print("Copia sin modificar guardada para comparación final:", df_original.shape)

Origen     : Hugging Face · scikit-learn/iris
Split      : train
Dimensiones: 150 filas x 6 columnas
Columnas   : Id, SepalLengthCm, SepalWidthCm, PetalLengthCm, PetalWidthCm, Species

Copia sin modificar guardada para comparación final: (150, 6)


## 3. Transformación y validación

Las operaciones de esta sección son las que pide el estado de los datos: verificación
de tipos, normalización de la columna de especie, control de valores faltantes,
duplicados y atípicos, y selección de columnas. Cada operación reporta cuántos
registros afectó; si un control no encuentra problemas, también se informa.

### 3.1 Verificación de tipos

In [2]:
df = df_original.copy()

# Tipos esperados según la documentación del dataset
esperados = {
    "Id": "int64",
    "SepalLengthCm": "float64",
    "SepalWidthCm": "float64",
    "PetalLengthCm": "float64",
    "PetalWidthCm": "float64",
    "Species": "object",
}
revision_tipos = pd.DataFrame({
    "tipo_actual": df.dtypes.astype(str),
    "tipo_esperado": pd.Series(esperados),
})
revision_tipos["ok"] = revision_tipos["tipo_actual"] == revision_tipos["tipo_esperado"]
revision_tipos

,tipo_actual,tipo_esperado,ok
Id,int64,int64,True
SepalLengthCm,float64,float64,True
SepalWidthCm,float64,float64,True
PetalLengthCm,float64,float64,True
PetalWidthCm,float64,float64,True
Species,object,object,True


Las seis columnas llegan con el tipo correcto, así que no hace falta ninguna
conversión. En cuanto al formato, la columna `Species` trae los valores con el
prefijo `Iris-` y capitalización propia del dataset original; se normaliza más
adelante en la sección 3.2.

### 3.2 Renombrar columnas y normalizar la especie

Las columnas pasan a español en formato `snake_case`, con la unidad incluida en
el nombre, igual como se hizo en la entrega anterior. La especie pierde el
prefijo `Iris-` y queda en minúscula.

In [3]:
renombres = {
    "Id": "id_flor",
    "SepalLengthCm": "largo_sepalo_cm",
    "SepalWidthCm": "ancho_sepalo_cm",
    "PetalLengthCm": "largo_petalo_cm",
    "PetalWidthCm": "ancho_petalo_cm",
    "Species": "especie",
}
df = df.rename(columns=renombres)

valores_antes = df["especie"].copy()
df["especie"] = df["especie"].str.strip().str.replace("Iris-", "", regex=False).str.lower()
celdas_cambiadas = int((valores_antes != df["especie"]).sum())

print(f"Columnas renombradas: {len(renombres)} (afecta a las {len(df)} filas)")
print(f"Valores de especie normalizados: {celdas_cambiadas} de {len(df)}")
print()
print(df["especie"].value_counts())

Columnas renombradas: 6 (afecta a las 150 filas)
Valores de especie normalizados: 150 de 150

especie
setosa        50
versicolor    50
virginica     50
Name: count, dtype: int64


### 3.3 Valores faltantes

In [4]:
nulos = df.isna().sum()
print("Valores nulos por columna:")
print(nulos.to_string())
print()
total_nulos = int(nulos.sum())
if total_nulos == 0:
    print("El control no detectó faltantes: no se aplica ningún tratamiento.")

Valores nulos por columna:
id_flor            0
largo_sepalo_cm    0
ancho_sepalo_cm    0
largo_petalo_cm    0
ancho_petalo_cm    0
especie            0

El control no detectó faltantes: no se aplica ningún tratamiento.


### 3.4 Duplicados

In [5]:
medidas = ["largo_sepalo_cm", "ancho_sepalo_cm", "largo_petalo_cm", "ancho_petalo_cm"]

dup_exactos = int(df.duplicated().sum())
grupos_repetidos = df[df.duplicated(subset=medidas, keep=False)]

print(f"Duplicados exactos (todas las columnas): {dup_exactos}")
print(f"Filas dentro de grupos con las cuatro medidas idénticas: {len(grupos_repetidos)}")
print()
grupos_repetidos[["id_flor", *medidas, "especie"]].sort_values(medidas)

Duplicados exactos (todas las columnas): 0
Filas dentro de grupos con las cuatro medidas idénticas: 5



,id_flor,largo_sepalo_cm,ancho_sepalo_cm,largo_petalo_cm,ancho_petalo_cm,especie
9,10,4.9,3.1,1.5,0.1,setosa
34,35,4.9,3.1,1.5,0.1,setosa
37,38,4.9,3.1,1.5,0.1,setosa
101,102,5.8,2.7,5.1,1.9,virginica
142,143,5.8,2.7,5.1,1.9,virginica


Aparecen **dos grupos de medidas idénticas**: tres flores *setosa* (identificadores
10, 35 y 38) y dos *virginica* (102 y 143). En total 5 filas de 150 (3,3 %).

**Decisión: conservarlas.** Cada fila tiene su propio identificador, así que son
ejemplares distintos con medidas que coinciden a la precisión registrada (un
décimo de centímetro), algo plausible en naturaleza. Eliminarlas alteraría las
frecuencias del dataset sin corregir ningún error de captura, y el criterio del
proyecto es no borrar datos sin justificación firme.

### 3.5 Valores atípicos

Se aplica la regla de 1,5 × RIC (rango intercuartílico) a las cuatro medidas,
solo como control: los valores marcados se revisan, pero no se eliminan
automáticamente.

In [6]:
conteo_atipicos = {}
for columna in medidas:
    q1, q3 = df[columna].quantile([0.25, 0.75])
    ric = q3 - q1
    fuera = (df[columna] < q1 - 1.5 * ric) | (df[columna] > q3 + 1.5 * ric)
    conteo_atipicos[columna] = int(fuera.sum())

serie_atipicos = pd.Series(conteo_atipicos, name="valores_atipicos")
serie_atipicos.to_frame().T

,largo_sepalo_cm,ancho_sepalo_cm,largo_petalo_cm,ancho_petalo_cm
valores_atipicos,0,4,0,0


In [7]:
q1, q3 = df["ancho_sepalo_cm"].quantile([0.25, 0.75])
ric = q3 - q1
fuera = (df["ancho_sepalo_cm"] < q1 - 1.5 * ric) | (df["ancho_sepalo_cm"] > q3 + 1.5 * ric)
df.loc[fuera, ["id_flor", *medidas, "especie"]]

,id_flor,largo_sepalo_cm,ancho_sepalo_cm,largo_petalo_cm,ancho_petalo_cm,especie
15,16,5.7,4.4,1.5,0.4,setosa
32,33,5.2,4.1,1.5,0.1,setosa
33,34,5.5,4.2,1.4,0.2,setosa
60,61,5.0,2.0,3.5,1.0,versicolor


Solo `ancho_sepalo_cm` tiene valores fuera del rango típico: cuatro flores, tres
con sépalos estrechos (2,0 cm, dos de ellas *setosa* y una *versicolor*) y una con
sépalo ancho (4,4 cm, *setosa*). Son valores medibles y coherentes con la
variabilidad de la especie, no errores de digitación (ningún valor excede los
límites físicos razonables para una flor de iris). **Decisión: conservarlas** y
dejar el hallazgo registrado.

### 3.6 Selección de columnas

El identificador es un consecutivo sin valor descriptivo: se descarta. Las cuatro
medidas y la especie se conservan porque las preguntas de negocio se apoyan todas
en ellas.

In [8]:
df = df.drop(columns="id_flor")
print(f"Columnas descartadas: 1 ('id_flor') · filas afectadas: 0 (no se elimina ningún registro)")
print("Dimensiones tras la selección:", df.shape)

Columnas descartadas: 1 ('id_flor') · filas afectadas: 0 (no se elimina ningún registro)
Dimensiones tras la selección: (150, 5)


### 3.7 Comparación antes y después

In [9]:
def perfil(frame: pd.DataFrame) -> pd.Series:
    """Resumen estructural de un DataFrame, para comparar versiones."""
    return pd.Series({
        "filas": len(frame),
        "columnas": frame.shape[1],
        "columnas_numericas": len(frame.select_dtypes("number").columns),
        "columnas_de_texto": len(frame.select_dtypes(exclude="number").columns),
        "valores_nulos": int(frame.isna().sum().sum()),
        "duplicados_exactos": int(frame.duplicated().sum()),
    })

comparacion = pd.concat(
    [perfil(df_original).rename("antes"), perfil(df).rename("después")],
    axis=1,
)
comparacion

,antes,después
filas,150,150
columnas,6,5
columnas_numericas,5,4
columnas_de_texto,1,1
valores_nulos,0,0
duplicados_exactos,0,3


La transformación no eliminó ni una fila: las 150 flores siguen presentes. Lo que
cambió son los nombres de las columnas (6 en total), el formato de los 150 valores
de `especie` y el descarte del identificador. Los controles de nulos, duplicados y
atípicos quedaron documentados con su decisión correspondiente.

## 4. Enriquecimiento de la información

Las variables nuevas se calculan a partir de las medidas existentes; no se
combina otra tabla porque el dataset ya contiene lo que las preguntas necesitan
y ninguna fuente externa aportaría a una caracterización morfológica. Las dos
áreas y la razón ya se usaron con fines ilustrativos en la entrega anterior;
aquí se incorporan formalmente al análisis y se les suma la categoría de tamaño.

| Variable nueva | Fórmula o regla | Significado | Utilidad para las preguntas |
| --- | --- | --- | --- |
| `area_sepalo_cm2` | largo × ancho del sépalo | Superficie aproximada del sépalo (cm²) | Tamaño global de la flor vista de fuera (P1, P2) |
| `area_petalo_cm2` | largo × ancho del pétalo | Superficie aproximada del pétalo (cm²) | Medida compuesta que resume el tamaño del pétalo (P2, P4) |
| `ratio_petalo` | largo ÷ ancho del pétalo | Proporción entre las dimensiones del pétalo | Forma del pétalo, complementaria del tamaño (P2) |
| `tamano_flor` | Terciles de `area_petalo_cm2` | Categoría: Pequeña, Media o Grande | Agrupar ejemplares para la distribución de la P4 |

In [10]:
df["area_sepalo_cm2"] = (df["largo_sepalo_cm"] * df["ancho_sepalo_cm"]).round(2)
df["area_petalo_cm2"] = (df["largo_petalo_cm"] * df["ancho_petalo_cm"]).round(2)
df["ratio_petalo"] = (df["largo_petalo_cm"] / df["ancho_petalo_cm"]).round(2)

# Cortes por terciles: cada categoria reune cerca de un tercio de las flores
limites = df["area_petalo_cm2"].quantile([1 / 3, 2 / 3])
df["tamano_flor"] = pd.cut(
    df["area_petalo_cm2"],
    bins=[-float("inf"), limites.iloc[0], limites.iloc[1], float("inf")],
    labels=["Pequeña", "Media", "Grande"],
)

print("Cortes de tamaño (terciles del área del pétalo):")
print(f"  Pequeña:  área < {limites.iloc[0]:.2f} cm²")
print(f"  Media   :  {limites.iloc[0]:.2f} – {limites.iloc[1]:.2f} cm²")
print(f"  Grande  :  área > {limites.iloc[1]:.2f} cm²")
print()
print(df["tamano_flor"].value_counts())

Cortes de tamaño (terciles del área del pétalo):
  Pequeña:  área < 2.52 cm²
  Media   :  2.52 – 7.71 cm²
  Grande  :  área > 7.71 cm²

tamano_flor
Pequeña    50
Media      50
Grande     50
Name: count, dtype: int64


El DataFrame final queda con **9 columnas** (5 originales depuradas + 4 calculadas)
y las mismas 150 filas. La sección 6 exporta este DataFrame junto con el resumen
descriptivo del archivo `.txt`.

## 5. Respuestas a las preguntas de negocio

Cada pregunta se responde con una tabla exportada a `resultados/` y con una
interpretación escrita a partir de sus números.

### P1 · ¿Cómo se caracteriza morfológicamente cada especie?

In [11]:
filas_p1 = []
for especie, grupo in df.groupby("especie", observed=True):
    for medida in medidas:
        filas_p1.append({
            "especie": especie,
            "medida": medida,
            "media_cm": round(grupo[medida].mean(), 2),
            "mediana_cm": round(grupo[medida].median(), 2),
            "desviacion_cm": round(grupo[medida].std(), 2),
        })
tabla_p1 = pd.DataFrame(filas_p1)

# Vista compacta para leer en el cuaderno: medias por especie
vista_p1 = df.groupby("especie", observed=True)[medidas].mean().round(2)
vista_p1

,largo_sepalo_cm,ancho_sepalo_cm,largo_petalo_cm,ancho_petalo_cm
especie,,,,
setosa,5.01,3.42,1.46,0.24
versicolor,5.94,2.77,4.26,1.33
virginica,6.59,2.97,5.55,2.03


In [12]:
RESULTADOS_DIR.mkdir(exist_ok=True)
tabla_p1.to_csv(RESULTADOS_DIR / "tabla_p1_perfil_especies.csv", index=False, encoding="utf-8-sig")
print("Exportada:", RESULTADOS_DIR / "tabla_p1_perfil_especies.csv")

Exportada: C:\Users\elbab\OneDrive\Documentos\IU Digital\Proyecto Integrado III - Analítica de Datos\AE3\resultados\tabla_p1_perfil_especies.csv


**Respuesta.** Las tres especies forman un gradiente claro de tamaño. *Setosa* es
la más pequeña en pétalo (pétalo de 1,46 × 0,24 cm en promedio, un área de apenas
0,36 cm²), pero tiene el sépalo más ancho (3,42 cm). *Virginica* encabeza las
cuatro medidas menos el ancho del sépalo, y *versicolor* queda al centro en todo.
La media y la mediana de cada especie casi coinciden, señal de distribuciones sin
asimetrías fuertes.

**Limitación.** Las cifras describen las 50 flores por especie de esta muestra
histórica; no son parámetros de la especie en general, y el dataset no trae
información de contexto (hábitat, fecha, madurez de la flor) que permita afinar
la caracterización.

### P2 · ¿Las especies se diferencian más por el sépalo o por el pétalo?

Se compara, para cada medida, la variación dentro de cada especie (coeficiente de
variación) y la distancia entre las medias de las tres especies (diferencia
relativa sobre la media global).

In [13]:
medias = df.groupby("especie", observed=True)[medidas].mean()
desviaciones = df.groupby("especie", observed=True)[medidas].std()
cv = (desviaciones / medias * 100)

rango_medias = medias.max() - medias.min()
tabla_p2 = pd.DataFrame({
    "cv_setosa_pct": cv.loc["setosa"],
    "cv_versicolor_pct": cv.loc["versicolor"],
    "cv_virginica_pct": cv.loc["virginica"],
    "rango_entre_medias_cm": rango_medias,
    "diferencia_relativa_pct": (rango_medias / medias.mean() * 100),
}).round(2)
tabla_p2.to_csv(RESULTADOS_DIR / "tabla_p2_variacion_medidas.csv", index=False, encoding="utf-8-sig")
tabla_p2

,cv_setosa_pct,cv_versicolor_pct,cv_virginica_pct,rango_entre_medias_cm,diferencia_relativa_pct
largo_sepalo_cm,7.04,8.70,9.65,1.58,27.07
ancho_sepalo_cm,11.15,11.33,10.84,0.65,21.22
largo_petalo_cm,11.85,11.03,9.94,4.09,108.76
ancho_petalo_cm,43.94,14.91,13.56,1.78,148.67


**Respuesta.** Por el pétalo. La diferencia entre especies alcanza un 108,8 % del
valor medio en el largo del pétalo y un 148,7 % en el ancho, mientras que en las
dos medidas del sépalo apenas llega al 27,1 % y 21,2 %. Dicho de otro modo: dos
flores de especies distintas casi siempre se distinguen mirando el pétalo, y las
medidas del sépalo se cruzan entre especies con mucha más frecuencia. La única
medida con variación interna alta dentro de una especie es el ancho del pétalo de
*setosa* (43,9 %), que refleja lo diminuto de su valor promedio (0,24 cm) más que
una dispersión real.

**Limitación.** La comparación usa medias y no prueba clasificación alguna; para
afirmar qué variables conviene usar en un modelo identificatorio harían falta
técnicas de clasificación, fuera del alcance descriptivo de esta etapa.

### P3 · ¿Qué relación existe entre las medidas del sépalo y las del pétalo?

Se calcula la correlación de Pearson para los cuatro pares sépalo–pétalo, primero
sobre todo el dataset y luego dentro de cada especie.

In [14]:
pares = [
    ("largo_sepalo_cm", "largo_petalo_cm"),
    ("largo_sepalo_cm", "ancho_petalo_cm"),
    ("ancho_sepalo_cm", "largo_petalo_cm"),
    ("ancho_sepalo_cm", "ancho_petalo_cm"),
]

filas_p3 = []
for variable_sepalo, variable_petalo in pares:
    for alcance in ["Global", "setosa", "versicolor", "virginica"]:
        grupo = df if alcance == "Global" else df[df["especie"] == alcance]
        filas_p3.append({
            "par_de_variables": f"{variable_sepalo} × {variable_petalo}",
            "alcance": alcance,
            "correlacion_pearson": round(grupo[variable_sepalo].corr(grupo[variable_petalo]), 2),
            "n_flores": len(grupo),
        })
tabla_p3 = pd.DataFrame(filas_p3)
tabla_p3.to_csv(RESULTADOS_DIR / "tabla_p3_correlaciones.csv", index=False, encoding="utf-8-sig")

# Vista por alcance para leerla mejor
vista_p3 = tabla_p3.pivot(index="par_de_variables", columns="alcance", values="correlacion_pearson")
vista_p3 = vista_p3[["Global", "setosa", "versicolor", "virginica"]]
vista_p3

alcance,Global,setosa,versicolor,virginica
par_de_variables,,,,
ancho_sepalo_cm × ancho_petalo_cm,-0.36,0.28,0.66,0.54
ancho_sepalo_cm × largo_petalo_cm,-0.42,0.18,0.56,0.40
largo_sepalo_cm × ancho_petalo_cm,0.82,0.28,0.55,0.28
largo_sepalo_cm × largo_petalo_cm,0.87,0.26,0.75,0.86


**Respuesta.** Globalmente, el largo del sépalo acompaña al pétalo con fuerza
(r = 0,87 con el largo y r = 0,82 con el ancho), pero el ancho del sépalo se
comporta al revés (r = −0,42 y −0,36). Ese signo negativo es engañoso: dentro de
cada especie la relación vuelve a ser positiva (hasta 0,66 en *versicolor*). Lo
que ocurre es que *setosa*, la especie de sépalo más ancho, tiene los pétalos más
pequeños y arrastra la correlación global hacia abajo. Es un recordatorio de que
una relación calculada sobre todo el dataset puede invertirse al mirar por grupo.

**Limitación.** Correlación no es causalidad: las cuatro medidas varían juntas
con el tamaño general de la flor, y con 50 flores por especie los coeficientes
por especie tienen bastante incertidumbre. Sirven para orientar, no para
modelar.

### P4 · ¿Cómo se distribuyen los ejemplares según el tamaño de la flor?

Con la categoría `tamano_flor` definida por terciles del área del pétalo se
cuentan los ejemplares por categoría y se cruza con la especie.

In [15]:
tabla_p4 = (
    df["tamano_flor"].value_counts()
    .rename_axis("tamano_flor")
    .reset_index(name="flores")
)
tabla_p4["participacion_pct"] = (tabla_p4["flores"] / len(df) * 100).round(1)
tabla_p4.to_csv(RESULTADOS_DIR / "tabla_p4_categorias_tamano.csv", index=False, encoding="utf-8-sig")
tabla_p4

,tamano_flor,flores,participacion_pct
0,Pequeña,50,33.3
1,Media,50,33.3
2,Grande,50,33.3


In [16]:
cruce = pd.crosstab(df["especie"], df["tamano_flor"])
porcentaje_especie = pd.crosstab(df["especie"], df["tamano_flor"], normalize="index").mul(100).round(1)

resultado_cruce = cruce.astype(str) + " flores (" + porcentaje_especie.astype(str) + " %)"
resultado_cruce.to_csv(RESULTADOS_DIR / "tabla_p4_cruce_especie_tamano.csv", encoding="utf-8-sig")
resultado_cruce

tamano_flor,Pequeña,Media,Grande
especie,,,
setosa,50 flores (100.0 %),0 flores (0.0 %),0 flores (0.0 %)
versicolor,0 flores (0.0 %),47 flores (94.0 %),3 flores (6.0 %)
virginica,0 flores (0.0 %),3 flores (6.0 %),47 flores (94.0 %)


**Respuesta.** Cada categoría reúne exactamente 50 flores (un tercio del total,
como corresponde a los cortes por terciles). El cruce con la especie es el
hallazgo más nítido del análisis: la categoría *Pequeña* es 100 % *setosa*, y las
otras dos especies se reparten *Media* y *Grande* casi sin mezclarse (94 % de
*versicolor* en *Media* y 94 % de *virginica* en *Grande*; solo 6 flores quedan en
la categoría vecina). El área del pétalo, variable calculada, resume en una sola
cantidad la separación que la P2 dejó ver medida por medida.

**Limitación.** Los cortes de 2,52 y 7,71 cm² dependen de esta muestra: con otros
datos los terciles se moverían y alguna flor cambiaría de categoría. La categoría
es descriptiva, no una regla de identificación validada.

## 6. Exportación de archivos

El dataset preparado y todas las tablas de resultados se escriben en formato
`.csv`, y el resumen descriptivo del DataFrame final se genera con la clase
`DatasetSummarizer` y se guarda en `resultados/resumen_dataset_ae3.txt`. Todo se
regenera en cada ejecución del cuaderno.

In [17]:
DATOS_DIR.mkdir(exist_ok=True)
RESULTADOS_DIR.mkdir(exist_ok=True)

ruta_dataset = DATOS_DIR / "dataset_preparado.csv"
df.to_csv(ruta_dataset, index=False, encoding="utf-8-sig")

resumen = DatasetSummarizer(
    df,
    nombre="Iris transformado y enriquecido (AE3)",
    origen=f"Hugging Face · {TARGET.dataset_id} · split={TARGET.split}",
)
ruta_txt = resumen.save()

print(f"Dataset preparado : {ruta_dataset.relative_to(RUTA_BASE)} ({ruta_dataset.stat().st_size} bytes)")
print(f"Resumen .txt      : {ruta_txt.relative_to(RUTA_BASE)} ({ruta_txt.stat().st_size} bytes)")
print()
print("\n".join(ruta_txt.read_text(encoding="utf-8").splitlines()[:14]))

Dataset preparado : datos\dataset_preparado.csv (7475 bytes)
Resumen .txt      : resultados\resumen_dataset_ae3.txt (2773 bytes)

                                  RESUMEN DEL DATASET
Nombre    : Iris transformado y enriquecido (AE3)
Generado  : 2026-11-07 11:53:24
Origen    : Hugging Face · scikit-learn/iris · split=train

1. DIMENSIONES
----------------------------------------------------------------------------------------
Filas      : 150
Columnas   : 9
Memoria    : 17.2 KB
Duplicados : 3 filas (2.00 %)



## 7. Verificación final

Antes de cerrar, se comprueba que los archivos exportados coinciden con el
cuaderno y que los totales y porcentajes de las tablas son consistentes.

In [18]:
comprobaciones: list[tuple[str, bool]] = []

def comprobar(nombre: str, condicion: bool) -> None:
    comprobaciones.append((nombre, bool(condicion)))

comprobar("El dataset preparado conserva las 150 filas originales", len(df) == len(df_original))
comprobar("No hay valores nulos tras la transformación", int(df.isna().sum().sum()) == 0)
comprobar("El perfil de la P1 cubre las 3 especies × 4 medidas",
          tabla_p1.groupby("especie")["medida"].count().eq(len(medidas)).all())
comprobar("La participación de la P4 suma 100 %", abs(tabla_p4["participacion_pct"].sum() - 100) < 0.5)
comprobar("El cruce de la P4 cubre las 150 flores", int(cruce.values.sum()) == len(df))
comprobar("Cada categoría de tamaño reúne cerca de un tercio",
          tabla_p4["flores"].between(40, 60).all())

# Los CSV exportados se releen y se comparan con lo que hay en memoria
releido = pd.read_csv(DATOS_DIR / "dataset_preparado.csv")
comprobar("dataset_preparado.csv coincide con el DataFrame final", releido.shape == df.shape)

for nombre, ok in comprobaciones:
    print(("OK    " if ok else "FALLO ") + nombre)
print()
print(f"{sum(ok for _, ok in comprobaciones)} de {len(comprobaciones)} comprobaciones superadas")

print()
print("Archivos generados en esta ejecución:")
for ruta in sorted([*DATOS_DIR.glob("*.csv"), *RESULTADOS_DIR.glob("*.csv"), *RESULTADOS_DIR.glob("*.txt")]):
    print(f"  {ruta.relative_to(RUTA_BASE)}  ({ruta.stat().st_size} bytes)")

OK    El dataset preparado conserva las 150 filas originales
OK    No hay valores nulos tras la transformación
OK    El perfil de la P1 cubre las 3 especies × 4 medidas
OK    La participación de la P4 suma 100 %
OK    El cruce de la P4 cubre las 150 flores
OK    Cada categoría de tamaño reúne cerca de un tercio
OK    dataset_preparado.csv coincide con el DataFrame final

7 de 7 comprobaciones superadas

Archivos generados en esta ejecución:
  datos\dataset_preparado.csv  (7475 bytes)
  resultados\resumen_dataset_ae3.txt  (2773 bytes)
  resultados\tabla_p1_perfil_especies.csv  (538 bytes)
  resultados\tabla_p2_variacion_medidas.csv  (216 bytes)
  resultados\tabla_p3_correlaciones.csv  (907 bytes)
  resultados\tabla_p4_categorias_tamano.csv  (90 bytes)
  resultados\tabla_p4_cruce_especie_tamano.csv  (225 bytes)


Con las comprobaciones en verde, el dataset queda listo para la presentación
gráfica de la siguiente etapa: las tablas de `resultados/` ya contienen las
agregaciones que cada visualización necesitará.